In [1]:
import keras
import pandas as pd
import numpy as np
from hyperopt import Trials, STATUS_OK, fmin,hp,tpe 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import mlflow
from mlflow.models import infer_signature

In [3]:
mlflow.set_experiment("real_estate_price_prediction")
dataset = pd.read_csv("Real estate.csv")
dataset.drop(columns=["X5 latitude","X6 longitude","No"], inplace=True)
dataset.describe()

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,Y house price of unit area
count,414.000000,414.000000,414.000000,414.000000,414.000000
mean,2013.148971,17.712560,1083.885689,4.094203,37.980193
std,0.281967,11.392485,1262.109595,2.945562,13.606488
min,2012.667000,0.000000,23.382840,0.000000,7.600000
25%,2012.917000,9.025000,289.324800,1.000000,27.700000
50%,2013.167000,16.100000,492.231300,4.000000,38.450000
75%,2013.417000,28.150000,1454.279000,6.000000,46.600000
max,2013.583000,43.800000,6488.021000,10.000000,117.500000


In [4]:
y=dataset["Y house price of unit area"]
X=dataset.drop(columns=["Y house price of unit area"])
dataset.isnull().sum()

X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
Y house price of unit area                0
dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.18, random_state=42)

In [6]:
y_train=y_train.ravel()
y_test=y_test.ravel()

In [7]:
trainX, validx, triany, validy = train_test_split(X_train, y_train, test_size=0.18, random_state=35)
trainX

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores
348,2012.833,4.6,259.6607,6
301,2012.750,38.0,461.7848,0
62,2012.917,17.2,2175.8770,3
329,2013.000,13.6,4197.3490,0
121,2013.500,13.6,492.2313,5
...,...,...,...,...
368,2013.417,18.2,350.8515,1
40,2013.000,13.6,4082.0150,0
132,2013.167,26.6,482.7581,5
259,2013.083,17.7,837.7233,0


In [8]:
triany

array([ 53.7,  35.7,  27.7,  19.2,  48. ,  40.8,  25. ,  36.7,  57.1,
        47.9,  17.7,  42.5,  26.5,  32.5,  23.6,  44.8,  25.3,  36.2,
        18.8,  37.4,  43.5,  31.7,  41.5,   7.6,  55.1,  60.7,  48.2,
        44.2,  12.9,  31.3,  39.5,  45.3,  40.3,  51.8,  42.5,  24.7,
        71. ,  46.8,  42. ,  43.2,  36.8,  50.2,  23.2,  38.4,  40.6,
        34. ,  47.1,  78.3,  42.3,  40.6,  51.6,  25.3,  37.4,  44.9,
        46.4,  36.3,  34.2,  28.6,  50.4,  49.8,  32.1,  27. ,  40.9,
        43.4,  32.4,  38.4,  31.6,  44.3,  24.7,  43.9,  42. ,  42.1,
        39.6,  39.4,  34.2,  51.7,  22. ,  34.6,  18.8,  34.3,  34.4,
        30.9,  12.8,  70.1,  22.3,  42.9,  41.1,  11.6,  17.4,  21.4,
        56.8,  33.1,  40. ,  25.5,  46.6,  55.3,  48.2,  45.5,  37.9,
        52.2,  46. ,  22.8,  57.8,  37.4,  47. ,  39.4,  26.2,  21.3,
        27.3,  55.2,  40.2,  43.8,  40.6,  21.8,  19. ,  23. ,  29.5,
        42.2,  52.7,  45.7,  28.5,  48.6,  69.7,  15. ,  45.9,  46.6,
        30.1,  24.5,

In [9]:
signature = infer_signature(trainX.to_numpy().astype(np.float32), triany.astype(np.float32))

In [10]:
def training_model(params, epochs, trainX, triany, validx, validy, X_test, y_test):
    mean=np.mean(trainX,axis=0)
    var=np.var(trainX,axis=0)
    normalizer = keras.layers.Normalization(axis=-1)
    normalizer.adapt(trainX)
    model = keras.Sequential([
        keras.Input(shape=(trainX.shape[1],)),
        normalizer,
        keras.layers.Dense(128,activation='relu'),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params['lr'],
        momentum=params['momentum']
    ),
      loss='mean_squared_error',
      metrics=[keras.metrics.RootMeanSquaredError()]
    )

    with mlflow.start_run(nested=True):
        model.fit(trainX,triany,validation_data=(validx,validy),epochs=epochs,batch_size=64)
        eval_model = model.evaluate(validx,validy,batch_size=64)[1]
        mlflow.log_params(params)
        mlflow.log_metric("mse",eval_model)
        mlflow.tensorflow.log_model(model,'model',signature=signature)

        return {'loss': float(eval_model), 'status': STATUS_OK}

In [11]:
def objective(params):
    result = training_model(
        params,
        epochs=5,
        trainX=trainX,
        triany=triany,
        validx=validx,
        validy=validy,
        X_test=X_test,
        y_test=y_test
    )
    return result

In [12]:
param_space = {
    'lr': hp.loguniform('lr',np.log(1e-5),np.log(1e-1)),
    'momentum': hp.uniform('momentum',0.0,1.0)
}

In [14]:
mlflow.set_experiment("real_estate_price_prediction_dl")
with mlflow.start_run():
    trials=Trials()
    best=fmin(
        space=param_space,
        fn=objective,
        max_evals=4,
        trials=trials
    )
    best_sorted = sorted(trials.results, key=lambda x:x['loss'])[0]
    mlflow.log_params(best)
    mlflow.log_metric('mse',best_sorted['loss'])

    print('best params:',best)
    print("Best eval:",best_sorted)

Epoch 1/5                                            

5/5 [==============================] - 0s 21ms/step - loss: 1690.3812 - root_mean_squared_error: 41.1142 - val_loss: 1527.5040 - val_root_mean_squared_error: 39.0833

Epoch 2/5                                            

5/5 [==============================] - 0s 5ms/step - loss: 1681.8934 - root_mean_squared_error: 41.0109 - val_loss: 1517.5663 - val_root_mean_squared_error: 38.9560

Epoch 3/5                                            

5/5 [==============================] - 0s 5ms/step - loss: 1671.3649 - root_mean_squared_error: 40.8823 - val_loss: 1507.4614 - val_root_mean_squared_error: 38.8260

Epoch 4/5                                            

5/5 [==============================] - 0s 6ms/step - loss: 1660.8267 - root_mean_squared_error: 40.7532 - val_loss: 1497.4221 - val_root_mean_squared_error: 38.6965

Epoch 5/5                                            

5/5 [==============================] - 0s 6ms/step - loss: 1

INFO:tensorflow:Assets written to: C:\Users\kesha\AppData\Local\Temp\tmpi9343sf5\model\data\model\assets



Epoch 1/5                                                                     

5/5 [==============================] - 0s 22ms/step - loss: 13353310.0000 - root_mean_squared_error: 3654.2180 - val_loss: 57403008393813164032.0000 - val_root_mean_squared_error: 7576477184.0000

Epoch 2/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: nan - root_mean_squared_error: nan - val_loss: nan - val_root_mean_squared_error: nan

Epoch 3/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: nan - root_mean_squared_error: nan - val_loss: nan - val_root_mean_squared_error: nan

Epoch 4/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: nan - root_mean_squared_error: nan - val_loss: nan - val_root_mean_squared_error: nan

Epoch 5/5                                    

INFO:tensorflow:Assets written to: C:\Users\kesha\AppData\Local\Temp\tmpl_h62g5z\model\data\model\assets



Epoch 1/5                                                                     

5/5 [==============================] - 0s 22ms/step - loss: 1658.4381 - root_mean_squared_error: 40.7239 - val_loss: 1458.1542 - val_root_mean_squared_error: 38.1858

Epoch 2/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: 1594.9934 - root_mean_squared_error: 39.9374 - val_loss: 1393.8411 - val_root_mean_squared_error: 37.3342

Epoch 3/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: 1523.9165 - root_mean_squared_error: 39.0374 - val_loss: 1317.4961 - val_root_mean_squared_error: 36.2973

Epoch 4/5                                                                     

5/5 [==============================] - 0s 6ms/step - loss: 1436.6880 - root_mean_squared_error: 37.9037 - val_loss: 1223.1903 - val_root_mean_squared_error: 34.9741

Epoch 5/5      

INFO:tensorflow:Assets written to: C:\Users\kesha\AppData\Local\Temp\tmpwv0i2pdr\model\data\model\assets



Epoch 1/5                                                                     

5/5 [==============================] - 0s 22ms/step - loss: 1688.1626 - root_mean_squared_error: 41.0873 - val_loss: 1513.9442 - val_root_mean_squared_error: 38.9094

Epoch 2/5                                                                    

5/5 [==============================] - 0s 6ms/step - loss: 1659.2548 - root_mean_squared_error: 40.7340 - val_loss: 1468.3041 - val_root_mean_squared_error: 38.3185

Epoch 3/5                                                                    

5/5 [==============================] - 0s 6ms/step - loss: 1603.4221 - root_mean_squared_error: 40.0428 - val_loss: 1399.4373 - val_root_mean_squared_error: 37.4091

Epoch 4/5                                                                    

5/5 [==============================] - 0s 6ms/step - loss: 1523.0359 - root_mean_squared_error: 39.0261 - val_loss: 1306.0084 - val_root_mean_squared_error: 36.1387

Epoch 5/5         

INFO:tensorflow:Assets written to: C:\Users\kesha\AppData\Local\Temp\tmp9lwqasmd\model\data\model\assets



100%|██████████| 4/4 [00:18<00:00,  4.67s/trial, best loss: 33.2089729309082]
best params: {'lr': 0.0006097317591468288, 'momentum': 0.14782192039570785}
Best eval: {'loss': 38.56409454345703, 'status': 'ok'}
